# 第2回 宿題（発展）: 用語集の Q&A で LoRA 学習し、学習前後と「用語集の検索」を比べる

演習（架空 FAQ の形式学習）のノートブックをもとに、学習データを **講座の用語集から作った Q&A**（`scripts/make_sft_dataset.py` の出力 `glossary_faq.txt`）に差し替えます。評価用の質問と正解は自分で用意し、学習前後を比べたうえで、**用語集を検索してプロンプトに入れる仕組み**（RAG 的な方法）とも比べます。

**この宿題で確かめること**

1. 学習前のモデルは、講座に固有の用語（Pi、既定モデル、宿題の3レベルなど）を知らない（セクション 4）。
2. 用語集の Q&A で LoRA 学習すると、学習と同じ聞き方の質問には定義どおり答えられるようになるか。言い回しを変えた質問ではどうか（セクション 6）。
3. 用語集に無い語を聞かれたとき、「記載がありません」と言えるか、もっともらしく創作するか（セクション 6）。
4. 学習せずに用語集を検索してプロンプトに入れる方法と比べ、学習のメリットとデメリットは何か（セクション 7、8）。

**進め方**

| セクション | 内容 | 所要時間の目安（T4） |
| --- | --- | --- |
| 0〜1 | 環境、設定 | 約 2 分 |
| 2〜3 | 学習データ（`glossary_faq.txt`）の読み込み、評価データの用意 | 1 分 |
| 4 | モデル読み込みと学習前評価 | 約 4 分 |
| 5 | LoRA 学習 | 約 8〜10 分 |
| 6 | 学習後評価と比較 | 約 3 分 |
| 7 | 用語集の検索との比較（学習前・学習後の両方） | 約 5 分 |
| 8 | 結果の保存と考察 | — |

**使うもの**: Google Colab（T4 GPU、無料枠）、`glossary_faq.txt`（Codespaces で `uv run python scripts/make_sft_dataset.py > lectures/02/glossary_faq.txt` で作ったもの。セクション 2 でアップロードします）。モデルは演習と同じ `Qwen/Qwen3.5-0.8B`。

**読み方**: 演習と同じく、各セクションに「やること」「見るところ」「うまくいかないとき」を書いています。セルは上から順に実行してください。

## 0. 環境の確認とライブラリの導入

**やること**: GPU が割り当てられているかを確認し、ライブラリを動作確認済みの版に固定して導入します。

**見るところ**

- 1 つ目のセルに `Tesla T4` と表示されること。
- 2 つ目のセルの最後に、版の一覧（`transformers: 5.18.0` など）が表示されること。導入は 1 分ほどかかります。

**うまくいかないとき**

- 「GPU が見つかりません」と出る: メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選び、セルを実行し直します。混雑で割り当てられないときは、時間をおいて試します。
- pip の赤い `ERROR: pip's dependency resolver ...`（`cudf` や `pyarrow` に関するもの）は、Colab に最初から入っている別のパッケージとの食い違いの警告です。この演習には影響しないので、そのまま進めます。

In [1]:
import subprocess

try:
    gpu_info = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
        capture_output=True,
        text=True,
    ).stdout
except FileNotFoundError:  # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(
    gpu_info
    or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。"
)

In [2]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",  # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",  # LoRA の実装
    "trl": "1.14.2",  # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import accelerate
import datasets
import peft
import torch
import transformers
import trl

print(
    {
        m.__name__: m.__version__
        for m in [transformers, peft, trl, accelerate, datasets, torch]
    }
)

## 1. 設定

**やること**: 設定を辞書 `CONFIG` にまとめます。セクション 8 の `result.json` にそのまま記録されます。演習から変えたのは次の点です。

- `num_train_epochs: 5`: 学習データは 116 件（58 語 × 2 テンプレート）で、演習の 14 件より多いのでエポック数を減らします。バッチ 4 × 勾配累積 2 = 1 ステップ 8 件なので、1 エポック 15 ステップ、合計約 75 ステップです。
- `max_new_tokens: 200`: 用語集の定義は最長 212 文字あり、演習の 96 トークンでは途中で切れるので増やします。
- `add_unknown_examples`: `True` にすると「用語集に無い語 → 『用語集に記載がありません。』」という例を学習データに足します（セクション 2）。まずは `False`（`make_sft_dataset.py` の出力そのまま）で実行し、余裕があれば `True` で再学習して比べると考察の材料になります。変えたら **ランタイム → セッションを再起動** して上から実行します。

**見るところ**: 設定の JSON が表示されます。

In [15]:
import json
import re
import time
from pathlib import Path

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",  # 演習と同じモデル（Apache-2.0）
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # 演習と同じ重みに固定
    "seed": 42,
    "num_train_epochs": 5,  # 学習データ（116 件）を何周学習するか。1 周 15 ステップなので合計約 75 ステップ
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 200,  # 用語集の定義は最長 212 文字。演習の 96 では途中で切れるので増やす
    "gen": {"do_sample": False},  # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "add_unknown_examples": False,  # True にすると「用語集に無い語 → 記載がありません」の例を学習データに足す
    "work_dir": "/content/glossary_lora",  # 結果ファイルの保存先
    "drive_dir": "/content/drive/MyDrive/aie2026/glossary_lora",  # USE_DRIVE=True のときの保存先
}
WORK = Path(CONFIG["work_dir"])
WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = True  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

## 2. 学習データ（用語集の Q&A）の読み込み

**やること**: `scripts/make_sft_dataset.py` で作った `glossary_faq.txt` を読み込みます。各行は `{"messages": [{"role": "user", ...}, {"role": "assistant", ...}]}` の形で、1 語につき「「{用語}」とは何ですか？」「{用語}について簡単に説明してください。」の 2 問、答えは用語集の定義そのものです。

ファイルの渡し方は次のどちらかです。

- **アップロード**（既定）: `/content/glossary_faq.txt` が無ければ、セルの下にファイル選択ボタンが出るので、手元の `lectures/02/glossary_faq.txt` を選びます。左のファイル一覧に直接ドラッグして置いても構いません。
- **GitHub から取得**: 自分の fork に push 済みなら、`FAQ_URL` に raw の URL を入れると取りに行きます。

あわせて次の 2 つを作ります。

- `GLOSSARY`（用語 → 定義）: セクション 3 で評価の正解語句が定義に含まれるかを点検し、セクション 7 で用語集の検索に使います。1 つ目のテンプレートから用語名を取り出します。
- `train_records`: 学習に使う行。`add_unknown_examples=True` のときは、用語集に無い 6 語 × 2 テンプレートの「記載がありません」例（12 件）を足します。これらの語は評価の「用語集にない語」と重ねていません。

**見るところ**: `読み込んだ行: 116 | 用語: 58 | 学習に使う件数: 116` と、学習例の 1 件が表示されます。

**うまくいかないとき**: `messages の形が想定と違います` と出たら、ファイルが `make_sft_dataset.py` の出力そのものか確かめてください（空行は無視します）。

In [5]:
FAQ_FILE = Path("/content/glossary_faq.txt")  # scripts/make_sft_dataset.py の出力
# 自分の fork に push 済みなら raw の URL から取ってもよい
# 例: "https://raw.githubusercontent.com/<ユーザー名>/lecture-ai-engineering-2026/lec02/lectures/02/glossary_faq.txt"
FAQ_URL = "https://raw.githubusercontent.com/YutaroSuga/lecture-ai-engineering-2026/lec02/lectures/02/glossary_faq.txt"

if not FAQ_FILE.exists():
    if FAQ_URL:
        import urllib.request

        urllib.request.urlretrieve(FAQ_URL, FAQ_FILE)
    else:
        from google.colab import files

        uploaded = files.upload()  # ダイアログで glossary_faq.txt を選ぶ
        FAQ_FILE.write_bytes(next(iter(uploaded.values())))

records = [
    json.loads(line)
    for line in FAQ_FILE.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
for r in records:
    roles = [m["role"] for m in r["messages"]]
    assert roles == ["user", "assistant"], f"messages の形が想定と違います: {roles}"

# 用語 → 定義。make_sft_dataset.py の 1 つ目のテンプレート「「{term}」とは何ですか？」から用語名を取り出す
TERM_PATTERN = re.compile(r"^「(.+)」とは何ですか？$")
GLOSSARY = {}
for r in records:
    m = TERM_PATTERN.match(r["messages"][0]["content"])
    if m:
        GLOSSARY[m.group(1)] = r["messages"][1]["content"]

NOT_FOUND = "用語集に記載がありません。"
TEMPLATES = [
    "「{term}」とは何ですか？",
    "{term}について簡単に説明してください。",
]  # make_sft_dataset.py と同じ
# 用語集に無い語（add_unknown_examples=True のときだけ学習に使う）。評価の「用語集にない語」とは重ねない
UNKNOWN_TRAIN_TERMS = [
    "ベクトルデータベース",
    "チャンク分割",
    "Few-shotプロンプト",
    "期末レポート",
    "Codespaces prebuild",
    "評価ダッシュボード",
]

train_records = list(records)
if CONFIG["add_unknown_examples"]:
    for term in UNKNOWN_TRAIN_TERMS:
        assert term not in GLOSSARY, f"{term} は用語集にあります"
        for t in TEMPLATES:
            train_records.append(
                {
                    "messages": [
                        {"role": "user", "content": t.format(term=term)},
                        {"role": "assistant", "content": NOT_FOUND},
                    ]
                }
            )

print(
    f"読み込んだ行: {len(records)} | 用語: {len(GLOSSARY)} | 学習に使う件数: {len(train_records)}"
)
print("学習例の 1 件:", json.dumps(train_records[0], ensure_ascii=False))

## 3. 評価データ（自分で用意した質問と正解）

**やること**: 評価用の質問 25 問と、その正解を用意します。種類は 3 つです。

| 種類（`kind`） | 件数 | 内容 | 正解 |
| --- | --- | --- | --- |
| `same` 用語集の語 | 8 | 学習データと **同じ聞き方**（「「LoRA」とは何ですか？」など） | 定義の特徴的な語句を含む |
| `paraphrase` 言い換え | 10 | 用語集に答えがあるが、**学習データに無い言い回し**（「Piのライセンスは何ですか？」など）。用語名を含まない質問も混ぜる | 同上 |
| `unknown` 用語集にない語 | 7 | 一般的な用語（RAG、Transformer など）と、実在の語に似せた架空の語（「lecNN-end タグ」など） | 「用語集に記載がありません」と答える |

- 一般的な AI 用語と、講座に固有の語（Pi、既定モデル、宿題の3レベルなど）を混ぜています。固有の語はモデルが事前学習で知り得ないので、学習の効果が出やすいところです。
- `unknown` の正解を「記載がありません」にしているのは、この仕組みが **用語集に根拠がある答えだけを返す** 役割だからです。RAG のようにモデルが一般知識で正しく説明できる語でも、用語集に無ければ根拠のない回答として × にします。
- 正解の判定は語句の照合です。`must_include` の語句をすべて含めば ○（`|` で区切った語はどれか 1 つでよい）。大文字小文字と空白は無視します。語句が定義に実際に含まれるか、言い換えの質問が学習データと重複していないかを、下のセルで点検します。

質問や語句は自由に足したり変えたりして構いません。変えたら、このセル以降を上から実行し直します。

**見るところ**: `評価データ: same=8 paraphrase=10 unknown=7` と表示されること。点検に失敗するとエラーになります。

In [6]:
# kind: same = 学習と同じ聞き方 / paraphrase = 言い回しを変えた質問 / unknown = 用語集にない語
# term: 答えの根拠になる用語集の用語（unknown では、用語集に無いことを確かめる語）
# must_include: 答えに含まれるべき語句。「|」で区切った語はどれか 1 つを含めばよい
EVAL = [
    # --- 用語集にある語（学習データと同じ聞き方） ---
    {"kind": "same", "question": "「LoRA」とは何ですか？", "term": "LoRA", "must_include": ["低ランク"]},
    {"kind": "same", "question": "「トークン」とは何ですか？", "term": "トークン", "must_include": ["最小単位"]},
    {"kind": "same", "question": "「量子化」とは何ですか？", "term": "量子化", "must_include": ["ビット"]},
    {"kind": "same", "question": "「Codespaces secret」とは何ですか？", "term": "Codespaces secret", "must_include": ["環境変数"]},
    {"kind": "same", "question": "「Pi」とは何ですか？", "term": "Pi", "must_include": ["MIT"]},
    {"kind": "same", "question": "「既定モデル」とは何ですか？", "term": "既定モデル", "must_include": ["gemini-3.5-flash-lite"]},
    {"kind": "same", "question": "「宿題の3レベル」とは何ですか？", "term": "宿題の3レベル", "must_include": ["基礎", "実装", "発展"]},
    {"kind": "same", "question": "「lecNN-start タグ」とは何ですか？", "term": "lecNN-start タグ", "must_include": ["タグ", "ブランチ"]},
    # --- 言い回しを変えた質問（答えは用語集にあるが、学習データに無い聞き方） ---
    {"kind": "paraphrase", "question": "LoRAでは、元のモデルの重みをどう扱いますか？", "term": "LoRA", "must_include": ["固定"]},
    {"kind": "paraphrase", "question": "トークン数によって決まるものを挙げてください。", "term": "トークン", "must_include": ["料金|レート制限|コンテキストウィンドウ"]},
    {"kind": "paraphrase", "question": "AIE2026の演習で標準的に使うモデルは何ですか？", "term": "既定モデル", "must_include": ["gemini-3.5-flash-lite"]},
    {"kind": "paraphrase", "question": "Piのライセンスは何ですか？", "term": "Pi", "must_include": ["MIT"]},
    {"kind": "paraphrase", "question": "APIキーは、どの環境変数から読み込みますか？", "term": "APIキー", "must_include": ["GEMINI_API_KEY"]},
    {"kind": "paraphrase", "question": "宿題はいくつ取り組めば、その回は完了になりますか？", "term": "宿題の3レベル", "must_include": ["1つ|一つ|ひとつ"]},
    {"kind": "paraphrase", "question": "前の回の演習が終わっていなくても、次の回の演習を始める方法はありますか？", "term": "lecNN-start タグ", "must_include": ["start"]},
    {"kind": "paraphrase", "question": "Pydantic AIで、ツールの実行前に人の承認を挟むにはどう指定しますか？", "term": "Human-in-the-loop", "must_include": ["requires_approval"]},
    {"kind": "paraphrase", "question": "過学習していることには、どうやって気づけますか？", "term": "過学習", "must_include": ["検証"]},
    {"kind": "paraphrase", "question": "temperatureを低くすると、出力はどうなりますか？", "term": "temperature", "must_include": ["安定"]},
    # --- 用語集にない語（正解は「用語集に記載がありません」） ---
    {"kind": "unknown", "question": "「RAG（検索拡張生成）」とは何ですか？", "term": "RAG（検索拡張生成）", "must_include": []},
    {"kind": "unknown", "question": "「エンベディング」とは何ですか？", "term": "エンベディング", "must_include": []},
    {"kind": "unknown", "question": "「Transformer」とは何ですか？", "term": "Transformer", "must_include": []},
    {"kind": "unknown", "question": "DPOについて簡単に説明してください。", "term": "DPO", "must_include": []},  # 学習と同じテンプレートで、語だけ未知
    {"kind": "unknown", "question": "「lecNN-end タグ」とは何ですか？", "term": "lecNN-end タグ", "must_include": []},  # 実在の「lecNN-start タグ」に似せた架空の語
    {"kind": "unknown", "question": "「Codespaces token」とは何ですか？", "term": "Codespaces token", "must_include": []},  # 「Codespaces secret」に似せた架空の語
    {"kind": "unknown", "question": "AIE2026の中間試験について説明してください。", "term": "中間試験", "must_include": []},  # 講座に固有だが用語集に無い
]


def normalize(text: str) -> str:
    """照合用: 大文字小文字と空白（全角含む）を無視する"""
    return text.casefold().replace(" ", "").replace("　", "")


# 点検: same は学習データと同じ質問、それ以外は学習データに無い質問。正解の語句は根拠の定義に実際に含まれる
train_questions = {r["messages"][0]["content"] for r in train_records}
for e in EVAL:
    assert (e["question"] in train_questions) == (e["kind"] == "same"), (
        f"学習データとの重複の扱いが kind と合いません: {e['question']}"
    )
    if e["kind"] == "unknown":
        assert e["term"] not in GLOSSARY, f"{e['term']} は用語集にあります"
        continue
    definition = normalize(GLOSSARY[e["term"]])
    for kw in e["must_include"]:
        assert any(normalize(alt) in definition for alt in kw.split("|")), (
            f"「{kw}」が {e['term']} の定義にありません"
        )
assert len({e["question"] for e in EVAL}) == len(EVAL), "評価の質問が重複しています"

KIND_LABEL = {
    "same": "用語集の語（学習と同じ聞き方）",
    "paraphrase": "言い換え",
    "unknown": "用語集にない語",
}
print(
    "評価データ:",
    " ".join(f"{k}={sum(e['kind'] == k for e in EVAL)}" for k in KIND_LABEL),
)
_ = (WORK / "eval_set.json").write_text(
    json.dumps(EVAL, ensure_ascii=False, indent=2), encoding="utf-8"
)

## 4. モデルの読み込みと学習前評価

**やること**: 演習と同じく基盤モデルを量子化なし（32bit）で読み込み、評価の 25 問に答えさせて採点します。

プロンプト（`SYSTEM`）には「用語集の定義に沿って答え、用語集に無い語は『用語集に記載がありません。』と答える」と書きます。**学習データにも評価にも同じ `SYSTEM` を付けます**。`glossary_faq.txt` の行には system が無いので、学習時に `to_chat` で先頭に足します（セクション 5）。学習の前後と検索ありの条件で、プロンプトの指示を揃えるためです。

**採点のしかた**（以降のセクションでも同じ関数を使います）

| 種類 | 判定 | 意味 |
| --- | --- | --- |
| 用語集の語・言い換え | `○ 正解` | 正解の語句をすべて含む |
| | `× 記載なしと回答（逃げ）` | 用語集にあるのに「記載がありません」と答えた |
| | `× 誤り・語句不足` | 答えたが、正解の語句が足りない（別の内容、創作など） |
| 用語集にない語 | `○ 記載なしと回答` | 「記載がありません」などと答えた |
| | `× 断定（創作）` | 用語集に無い語を説明してしまった |

「記載がありません」の判定は決まった言い回し（`REFUSAL_MARKERS`）の照合なので、「申し訳ありませんが分かりません」のような別の言い方は拾えません。語句の照合も機械的なので、**表の出力を自分の目で読んで、判定が妥当か確かめてください**。

**見るところ**

- 要約の 1 行: `acc_all`（全体の正解率）と、種類ごとの `acc_same`・`acc_paraphrase`・`acc_unknown`。
- 質問ごとの表: 学習前は、一般的な用語はそれらしく説明できても定義の語句と合わない、講座に固有の語は知らずに創作する、用語集にない語も説明してしまう、といった出力が多いはずです。

出ても問題ない警告（`unauthenticated requests to the HF Hub`、`falling back to its reference PyTorch implementation`）は演習と同じです。

In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else ("mps" if torch.backends.mps.is_available() else "cpu")
)
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32
).to(DEVICE)
model.config.use_cache = True
print("loaded:", CONFIG["base_model"], "| device:", DEVICE)

# 学習データ・評価・検索ありのすべてで同じ指示を使う
SYSTEM = f"あなたは講座 AIE2026 の用語集に基づいて答えるアシスタントです。用語集にある用語は、その定義に沿って簡潔に答えてください。用語集に無い用語を聞かれたら、推測で説明せず『{NOT_FOUND}』とだけ答えてください。"


def build_messages(question, context=None):
    # context: 用語集の検索結果（セクション 7）。None なら質問だけを渡す（学習データと同じ形）
    user = question if context is None else f"用語集の検索結果:\n{context}\n\n質問: {question}"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]


def generate(model, question, context=None):
    # enable_thinking=False: 思考ブロックを空にして回答だけを生成させる
    prompt = tok.apply_chat_template(
        build_messages(question, context),
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=CONFIG["max_new_tokens"],
            pad_token_id=tok.pad_token_id,
            **CONFIG["gen"],
        )
    return tok.decode(
        out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    ).strip()

In [8]:
import pandas as pd

pd.set_option("display.max_colwidth", None)  # 出力の原文を省略せずに表示する

REFUSAL_MARKERS = ["記載がありません", "記載されていません", "見つかりません", "載っていません"]
OUTCOME_LABEL = {
    "correct": "○ 正解",
    "abstained": "○ 記載なしと回答",
    "refused": "× 記載なしと回答（逃げ）",
    "wrong": "× 誤り・語句不足",
    "asserted": "× 断定（創作）",
}
SUMMARY_COLS = ["n", "acc_all", "acc_same", "acc_paraphrase", "acc_unknown", "seconds"]


def has_keywords(text, must_include):
    t = normalize(text)
    return all(any(normalize(alt) in t for alt in kw.split("|")) for kw in must_include)


def classify(raw, item):
    refused = any(m in raw for m in REFUSAL_MARKERS)
    if item["kind"] == "unknown":
        return "abstained" if refused else "asserted"
    if refused:
        return "refused"
    return "correct" if has_keywords(raw, item["must_include"]) else "wrong"


def evaluate(model, label, use_search=False):
    """評価の 25 問を生成して採点し、要約と質問ごとの結果を返す。原文は eval_<label>.json に保存する"""
    rows = []
    t0 = time.time()
    for item in EVAL:
        context = search_glossary(item["question"]) if use_search else None
        raw = generate(model, item["question"], context)
        outcome = classify(raw, item)
        rows.append(
            {
                "question": item["question"],
                "kind": item["kind"],
                "context": context,
                "raw": raw,
                "outcome": outcome,
                "ok": outcome in ("correct", "abstained"),
            }
        )
    summary = {"label": label, "n": len(rows)}
    summary["acc_all"] = round(sum(r["ok"] for r in rows) / len(rows), 3)
    for kind in KIND_LABEL:
        sub = [r for r in rows if r["kind"] == kind]
        summary[f"acc_{kind}"] = round(sum(r["ok"] for r in sub) / len(sub), 3)
    summary["seconds"] = round(time.time() - t0, 1)
    (WORK / f"eval_{label}.json").write_text(
        json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows


before_summary, before_rows = evaluate(model, "before")
display(
    pd.DataFrame(
        [
            {
                "種類": KIND_LABEL[r["kind"]],
                "質問": r["question"],
                "学習前の出力": r["raw"],
                "判定": OUTCOME_LABEL[r["outcome"]],
            }
            for r in before_rows
        ]
    )
)

## 5. LoRA 学習（T4 で 8〜10 分）

**やること**: `train_records`（116 件）を chat 形式に直し、演習と同じ LoRA の設定で 5 エポック学習します。

- `glossary_faq.txt` の各行は `messages`（user / assistant）の形なので、`to_chat` では先頭に `SYSTEM` を足すだけです。評価と同じ `build_messages` を使うので、学習時と評価時の入力の形が揃います。
- 演習と違い `eval_dataset`（valid）は渡しません。用語集のすべての語を学習させ、汎化はセクション 3 の言い換えの質問で確かめます。
- バッチ 4 × 勾配累積 2 = 1 ステップ 8 件。116 件なので 1 エポック 15 ステップ、5 エポックで約 75 ステップです。

**見るところ**

- `trainable params: ... || trainable%: ...`: 演習と同じく全体の数 % だけを学習します。
- `Training Loss`（10 ステップごと）: 下がっていくこと。最後に表示する最終 loss は、セクション 8 で `result.json` に記録します。
- `学習時間: ... 秒 | ステップ数: ... | エポック: 5.0`

**うまくいかないとき**: 途中で止まった場合は、演習の 4b と同じく、セクション 0〜4 を実行し直してから `trainer.train()` を `trainer.train(resume_from_checkpoint=True)` に書き換えて再開します（チェックポイントは 25 ステップごとに `/content/glossary_lora/ckpt/` に保存）。時間が足りないときは `CONFIG["num_train_epochs"]` を 3 に減らします。

In [9]:
from datasets import Dataset
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer


def to_chat(record):
    # glossary_faq.txt の行（user / assistant）の先頭に、評価と同じ SYSTEM を足す
    user, assistant = record["messages"]
    return {"messages": build_messages(user["content"]) + [assistant]}


train_ds = Dataset.from_list([to_chat(r) for r in train_records])

model.config.use_cache = False
# Qwen3.5 は 24 層のうち 18 層が線形注意で、q/k/v/o_proj を持つのは残りの 6 層。gate/up/down_proj（MLP）は全層にある
lora = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)
sft = SFTConfig(
    output_dir=str(WORK / "ckpt"),
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=CONFIG["lr"],
    lr_scheduler_type="cosine",
    warmup_steps=CONFIG["warmup_steps"],
    logging_steps=10,
    save_steps=25,
    save_total_limit=2,
    max_length=CONFIG["max_seq_len"],
    packing=False,
    report_to="none",
    seed=CONFIG["seed"],
)
trainer = SFTTrainer(
    model=model,
    args=sft,
    train_dataset=train_ds,
    peft_config=lora,
    processing_class=tok,
)
trainer.model.print_trainable_parameters()
t0 = time.time()
trainer.train()  # 途中で止まった場合は trainer.train(resume_from_checkpoint=True)
TRAIN_SECONDS = round(time.time() - t0)
FINAL_LOSS = [h["loss"] for h in trainer.state.log_history if "loss" in h][-1]
print(
    f"学習時間: {TRAIN_SECONDS} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f} | 最終 loss: {FINAL_LOSS:.3f}"
)
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil

    shutil.copytree(
        ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True
    )
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # 学習モードのままだと dropout が効き、評価の出力が揺れる

## 6. 学習後評価（学習前と同じ質問・同じ生成設定）

**やること**: 学習後のモデル `tuned` で、学習前と同じ 25 問を同じ設定で評価し、種類ごとに前後を並べます。

**見るところ**

- 比較表: 種類ごとの正解率がどう変わったか。
  - `acc_same`（学習と同じ聞き方）: 定義をそのまま覚えたかの確認です。上がるのが期待値です。
  - `acc_paraphrase`（言い換え）: 覚えた定義を、聞き方が違っても引き出せるか。`same` との差が、学習で身についたのが「質問文と答えの対応」なのか「知識」なのかの手がかりです。
  - `acc_unknown`（用語集にない語）: `add_unknown_examples=False` なら、学習データには「答える」例しか無いので、何を聞かれても定義らしい文で答えるようになる（創作が増える）ことがあります。
- `退行`: 学習前は ○ だったのに学習後に × になった質問。
- 質問ごとの表: 言い換えで × になった質問は、別の用語の定義を返したのか、内容は合っているが語句が足りないのかを読み分けます。用語集にない語で × の場合、似た語（「lecNN-start タグ」「Codespaces secret」など）の定義を流用していないかを見ます。

演習と同じく、GPU 上の学習は完全には再現しないので、どの質問が ○ になるかは実行ごとに少し変わることがあります。

In [10]:
after_summary, after_rows = evaluate(tuned, "after")
display(pd.DataFrame([before_summary, after_summary]).set_index("label")[SUMMARY_COLS])
regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["ok"] and not a["ok"]]
print("退行:", regressed)

display(
    pd.DataFrame(
        [
            {
                "種類": KIND_LABEL[b["kind"]],
                "質問": b["question"],
                "学習前の出力": b["raw"],
                "学習前": OUTCOME_LABEL[b["outcome"]],
                "学習後の出力": a["raw"],
                "学習後": OUTCOME_LABEL[a["outcome"]],
            }
            for b, a in zip(before_rows, after_rows)
        ]
    )
)

## 7. 用語集を検索する仕組みとの比較

**やること**: モデルを学習させる代わりに、**質問に出てくる用語を用語集から探し、その定義をプロンプトに入れて答えさせる**方法（RAG 的な方法）で同じ 25 問を評価します。次の 2 条件を測ります。

| ラベル | モデル | 検索 | 意味 |
| --- | --- | --- | --- |
| `before_search` | 学習前（adapter を無効化） | あり | 学習せず、検索だけで答える |
| `after_search` | 学習後 | あり | 学習と検索を組み合わせる |

**検索のしかた**（`search_glossary`）: 第3回の `app/tools.py` の `search_glossary` と同じく用語名の **文字列照合** ですが、検索語を 1 つ渡すのではなく、質問文全体から用語名を探します。質問文に用語名（括弧や `/` で区切った別名も含む。例: 「LLM（大規模言語モデル）」は「LLM」「大規模言語モデル」でも当たる）が含まれる用語を、長い名前で当たったものから最大 2 件返します。英字の名前は単語の境界で照合します（「API」の中の「pi」で「Pi」に当たらないように）。見つからなければ「該当する用語は見つかりませんでした。」を渡します。

- 意味の近さで探す（埋め込みベクトルを使う）検索ではないので、**用語名を含まない言い換え**（「AIE2026の演習で標準的に使うモデルは何ですか？」など）では何も見つからず、モデルは「記載がありません」と答えるはずです。これも検索の仕組みの性質として結果に含めます。
- 用語集に無い語は検索で何も見つからないので、プロンプトに「見つからない」と明示されます。

**見るところ**

- 4 条件（`before`、`after`、`before_search`、`after_search`）の比較表。種類ごとに、学習と検索のどちらが効いているか。
- 質問ごとの表の `検索で見つかった用語`: 検索が当たらなかった質問で何が起きたか、当たったのに × の質問ではモデルが定義を正しく使えたか。
- 列 `seconds`: 検索ありは定義の分だけ入力が長くなります。

In [11]:
MAX_RESULTS = 2  # 1 回の検索で返す用語の数


def _names(term):
    # 「LLM（大規模言語モデル）」は LLM・大規模言語モデル でも、「ツール（Tool use / Function calling）」は / で区切った語でも引ける
    parts = [term, *re.split(r"[（）/]", term)]
    return {normalize(p) for p in parts if len(normalize(p)) >= 2}


def _mentions(question, name):
    if name.isascii():  # 英字の名前は単語の境界で照合する（「api」の中の「pi」に当たらないように）
        return re.search(rf"(?<![a-z0-9]){re.escape(name)}(?![a-z0-9])", question) is not None
    return name in question


def search_glossary(question):
    """質問文に名前が出てくる用語を、長い名前で当たったものから MAX_RESULTS 件返す"""
    q = normalize(question)
    hits = []
    for term, definition in GLOSSARY.items():
        matched = [n for n in _names(term) if _mentions(q, n)]
        if matched:
            hits.append((max(len(n) for n in matched), term, definition))
    if not hits:
        return "該当する用語は見つかりませんでした。"
    hits.sort(key=lambda h: -h[0])
    return "\n".join(f"- {term}: {definition}" for _, term, definition in hits[:MAX_RESULTS])


def hit_terms(context):
    return ", ".join(line[2:].split(": ")[0] for line in context.splitlines() if line.startswith("- ")) or "（なし）"


# 学習前の出力は adapter を一時的に無効にして取る
with tuned.disable_adapter():
    before_search_summary, before_search_rows = evaluate(tuned, "before_search", use_search=True)
after_search_summary, after_search_rows = evaluate(tuned, "after_search", use_search=True)

all_summaries = [before_summary, after_summary, before_search_summary, after_search_summary]
display(pd.DataFrame(all_summaries).set_index("label")[SUMMARY_COLS])
display(
    pd.DataFrame(
        [
            {
                "種類": KIND_LABEL[b["kind"]],
                "質問": b["question"],
                "検索で見つかった用語": hit_terms(bs["context"]),
                "学習前": OUTCOME_LABEL[b["outcome"]],
                "学習後": OUTCOME_LABEL[a["outcome"]],
                "検索のみ": OUTCOME_LABEL[bs["outcome"]],
                "検索のみの出力": bs["raw"],
                "学習＋検索": OUTCOME_LABEL[as_["outcome"]],
            }
            for b, a, bs, as_ in zip(before_rows, after_rows, before_search_rows, after_search_rows)
        ]
    )
)

## 8. 結果の保存と考察

**やること**: 設定、ライブラリの版、学習データの情報、4 条件の要約、退行した質問を `result.json` にまとめます。

**提出するもの**（`/content/glossary_lora` にあります。左のファイル一覧で右クリック → ダウンロード）

- `result.json`、`eval_before.json`、`eval_after.json`、`eval_before_search.json`、`eval_after_search.json`、`eval_set.json`（自分で用意した評価データ）
- `lectures/02/answer.md` の「宿題」欄: 取り組んだレベル（発展）、学習データ（`glossary_faq.txt`、116 件）と評価データの作り方、4 条件の比較表、考察

**考察の観点**（自分の結果の数字と出力を根拠に書きます）

| 観点 | 結果から確かめること |
| --- | --- |
| 覚えたことを引き出せるか | `same` と `paraphrase` の差。学習と同じ聞き方でしか答えられないなら、何を学習したと言えるか |
| 知らないことへの判断 | `unknown` が学習でどう変わったか。似た語の定義を流用した例はあるか。`add_unknown_examples=True` にすると変わるか |
| 検索の弱点 | 用語名を含まない言い換えで、検索が何も返さなかった質問はどれか。学習後のモデルはそれに答えられたか |
| 根拠と更新 | 用語集の定義が変わったとき、それぞれの方法で何をし直す必要があるか。答えの根拠（どの定義を使ったか）を示せるのはどちらか |
| コスト | 学習時間（`train_seconds`）、評価時間（`seconds`）、プロンプトの長さ。学習の手間と、毎回の入力が長くなる手間のどちらが重いか |
| 組み合わせ | `after_search` は `before_search` より良いか。学習と検索を役割分担させるなら、それぞれに何を任せるか |

In [ ]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし）",
    "data": {
        "train_file": FAQ_FILE.name,
        "n_records": len(records),
        "n_terms": len(GLOSSARY),
        "n_train": len(train_records),
        "n_eval": {k: sum(e["kind"] == k for e in EVAL) for k in KIND_LABEL},
    },
    "train_seconds": TRAIN_SECONDS,
    "train_steps": trainer.state.global_step,
    "final_loss": FINAL_LOSS,
    "summaries": all_summaries,
    "regressed": regressed,
    "executed_training_myself": True,  # 講師や他の人の結果を使った場合は False にする
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(result, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil

    for name in [
        "result.json",
        "eval_set.json",
        "eval_before.json",
        "eval_after.json",
        "eval_before_search.json",
        "eval_after_search.json",
    ]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("Drive に保存しました:", CONFIG["drive_dir"])